In [ ]:
import pathlib
import re

import numpy as np
import pandas as pd
import ydata_profiling

In [ ]:
SIMULATION_LENGTH="simulation_length"
EXPOSED="exposed"
NOT_EXPOSED="not_exposed"
PEAK_INFECTED="peak_infected"
PEAK_ITERATION="peak_iteration"
PROTOCOL = "protocol"
P = "p"
NETWORK = "network"
ACTOR = "actor"

In [ ]:
def get_simulation_params(file_name: pathlib.Path) -> tuple[str, str, str]:
    pattern = r"proto-(OR|AND)--p-([\d.]+)--net-(.*)\.csv"
    return re.match(pattern, file_name.name).groups()


def read_csv(file_path: pathlib.Path) -> pd.DataFrame:
    proto, p, net = get_simulation_params(file_path)
    file_df = pd.read_csv(file_path)
    file_df["network"] = net
    file_df["protocol"] = proto
    file_df["p"] = p
    return file_df

In [ ]:
raw_csvs = []
for idx, file_path in enumerate(
    pathlib.Path("ns-data-sources/spreading_potentials/multi_layer_networks/small_real").glob("*.csv")
):
    print(f"processing {idx}th file: {file_path.name}")
    raw_csvs.append(read_csv(file_path))

raw_csv = pd.concat(raw_csvs, axis=0, ignore_index=True)
print(f"\n\navailable networks: {raw_csv['network'].unique()}\n\n")
raw_csv.head()

In [ ]:
OUT_ROOT_DIR = "eda"
ANALYSED_NETWORK = "aucs"

In [ ]:
result_grouped = raw_csv.loc[raw_csv[NETWORK] == ANALYSED_NETWORK].groupby(
    by=[NETWORK, PROTOCOL, P, ACTOR]
)
result_mean = result_grouped.mean()
result_std = result_grouped.std()

result_mean.head()

In [ ]:
out_dir = pathlib.Path(f"{OUT_ROOT_DIR}/{ANALYSED_NETWORK}")
out_dir.mkdir(exist_ok=True, parents=True)

ydata_profiling.ProfileReport(result_mean, title=f"{ANALYSED_NETWORK} mean").to_file(f"{out_dir}/mean.html")
ydata_profiling.ProfileReport(result_std, title=f"{ANALYSED_NETWORK} std").to_file(f"{out_dir}/std.html")

In [ ]:
result_all = pd.merge(result_mean, result_std, left_index=True, right_index=True, suffixes=("_avg", "_std")).sort_index().reset_index(ACTOR)
result_all.head()

In [ ]:
import matplotlib

from matplotlib.axes import Axes
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import pyplot as plt

matplotlib.use('inline')


def plot_ax(
    partial_result: pd.DataFrame, metric: str, y_lim: tuple[int, int], ax: Axes, top_k: int = 5
) -> None:
    top_k_pr = partial_result.nlargest(n=top_k, columns=f"{metric}_avg")
    ax.errorbar(
        x=partial_result[ACTOR],
        y=partial_result[f"{metric}_avg"],
        yerr=partial_result[f"{metric}_std"],
        fmt="o",
        color = "royalblue",
    )
    ax.errorbar(
        x=top_k_pr[ACTOR],
        y=top_k_pr[f"{metric}_avg"],
        yerr=top_k_pr[f"{metric}_std"],
        fmt="o",
        color = "orange",
    )
    ax.set_title(f"{metric}, top {top_k}: {list(top_k_pr[ACTOR].sort_values())}")
    ax.set_ylim(y_lim)


def plot_partial_result(
    partial_result: pd.DataFrame,
    suptitle: str,
    sl_range: tuple[int, int] = None,
    ex_range: tuple[int, int] = None,
    pi_range: tuple[int, int] = None,
    pt_range: tuple[int, int] = None,
):
    fig, ax = plt.subplots(nrows=1, ncols=4)
    fig.set_size_inches(w=16, h=2)
    for idx, (metric, lim) in enumerate(
        zip(
            [EXPOSED, SIMULATION_LENGTH, PEAK_INFECTED, PEAK_ITERATION],
            [ex_range, sl_range, pi_range, pt_range]
        )
    ):
        plot_ax(partial_result, metric, lim, ax[idx])
    fig.subplots_adjust(top=0.9)
    fig.suptitle(suptitle)
    fig.tight_layout()
    return fig


def plot_title_page(title: str):
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(16, 2))
    ax.set_visible(False)
    fig.suptitle(title, x=0.5, y=.5, fontsize = 15)
    return fig

In [ ]:
simulated_cases = {idx_name: list(idx_lvl) for idx_name, idx_lvl in zip(result_all.index.names, result_all.index.levels)}

pdf = PdfPages(out_dir / (f"plots.pdf"))

for network in simulated_cases[NETWORK]:

    # title page - network
    partial_fig = plot_title_page(network)
    partial_fig.savefig(pdf, format="pdf")
    plt.close(partial_fig)

    # obtain y axis values' range for this network
    maxx = result_all.loc[network].max()
    ex_range = [0, np.ceil(maxx[f"{EXPOSED}_avg"]).astype(int)]
    sl_range = [0, np.ceil(maxx[f"{SIMULATION_LENGTH}_avg"]).astype(int)]
    pi_range = [0, np.ceil(maxx[f"{PEAK_INFECTED}_avg"]).astype(int)]
    pt_range = [0, np.ceil(maxx[f"{PEAK_ITERATION}_avg"]).astype(int)]

    # mean result across the network
    partial_fig = plot_partial_result(
        partial_result=result_all.loc[network].groupby(ACTOR).mean().reset_index(),
        suptitle=r"$\bf{" + f"net-{network}" + "}$",
        sl_range=sl_range,
        ex_range=ex_range,
        pi_range=pi_range,
        pt_range=pt_range,
    )
    partial_fig.savefig(pdf, format="pdf")
    plt.close(partial_fig)

    for proto in simulated_cases[PROTOCOL]:

        # title page - protocol
        partial_fig = plot_title_page(proto)
        partial_fig.savefig(pdf, format="pdf")
        plt.close(partial_fig)

        # obtain y axis values' range for this network and protocol
        maxx = result_all.loc[network].loc[proto].max()
        ex_range = [0, np.ceil(maxx[f"{EXPOSED}_avg"]).astype(int)]
        sl_range = [0, np.ceil(maxx[f"{SIMULATION_LENGTH}_avg"]).astype(int)]
        pi_range = [0, np.ceil(maxx[f"{PEAK_INFECTED}_avg"]).astype(int)]
        pt_range = [0, np.ceil(maxx[f"{PEAK_ITERATION}_avg"]).astype(int)]

        # mean result across the network and protocol
        partial_fig = plot_partial_result(
            partial_result=result_all.loc[network].loc[proto].groupby(ACTOR).mean().reset_index(),
            suptitle=f"net-{network}--" + r"$\bf{" + f"proto-{proto}" + "}$",
            sl_range=sl_range,
            ex_range=ex_range,
            pi_range=pi_range,
            pt_range=pt_range,
        )
        partial_fig.savefig(pdf, format="pdf")
        plt.close(partial_fig)

        # plot result for all particular experiments
        for p in simulated_cases[P]:
            partial_fig = plot_partial_result(
                partial_result=result_all.loc[network].loc[proto].loc[p],
                suptitle=f"net-{network}--proto-{proto}--" + r"$\bf{" + f"p-{p}" + "}$",
                sl_range=sl_range,
                ex_range=ex_range,
                pi_range=pi_range,
                pt_range=pt_range,
            )
            partial_fig.savefig(pdf, format="pdf")
            plt.close(partial_fig)

pdf.close()